In [39]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, errors, thermal_relaxation_error
from qiskit.quantum_info import Pauli, state_fidelity
import numpy as np
import matplotlib.pyplot as plt

In [68]:
def create_bell_state_pair_with_delay(delay_duration):
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    # Insert an idle wait on both qubits
    qc.delay(delay_duration, 0, unit='us')
    qc.delay(delay_duration, 1, unit='us')
    qc.delay(delay_duration, 0, unit='us')
    qc.delay(delay_duration, 1, unit='us')
    return qc

In [69]:
def measure_in_basis(qc, basis, qubit, cbit):
    if basis == "Z":
        pass
    elif basis == "X":
        qc.h(qubit)
    elif basis == "Z+X":
        qc.ry(-np.pi/4, qubit)
    elif basis == "Z-X":
        qc.ry(np.pi/4, qubit)
    qc.measure(qubit, cbit)

In [70]:
def compute_correlation(counts, shots):
    return (
        counts.get("00", 0) + counts.get("11", 0)
        - counts.get("01", 0) - counts.get("10", 0)
    ) / shots

In [71]:
def make_noise_model(T1, T2, gate_time_us=1.0):
    noise_model = NoiseModel()
    err_1q = thermal_relaxation_error(T1, T2, gate_time_us)
    noise_model.add_all_qubit_quantum_error(err_1q, ["delay"])
    return noise_model

In [72]:
def run_chsh_for_pair(t_us, shots, T1, T2):
    noise_model = make_noise_model(T1, T2, gate_time_us=1.0)
    sim = AerSimulator(noise_model=noise_model)
    settings = [("Z", "Z+X"), ("Z", "Z-X"), ("X", "Z+X"), ("X", "Z-X")]
    e = []
    for basisA, basisB in settings:
        qc = QuantumCircuit(2, 2)
        qc.h(0)
        qc.cx(0, 1)
        qc.delay(t_us, 0, unit="us")
        qc.delay(t_us, 1, unit="us")
        measure_in_basis(qc, basisA, 0, 0)
        measure_in_basis(qc, basisB, 1, 1)
        r = sim.run(qc, shots=shots).result()
        e.append(compute_correlation(r.get_counts(), shots))
    return e[0] + e[1] + e[2] - e[3]


In [73]:
def compute_chsh_parameter_thermal(time_steps, shots, num_pairs, T1=50, T2=70):
    s_values = []
    all_s_values = []
    for t in time_steps:
        s_total = 0
        s_list = []
        for _ in range(num_pairs):
            s = run_chsh_for_pair(t, shots, T1, T2)
            s_total += s
            s_list.append(s)
        s_values.append(s_total / num_pairs)
        all_s_values.append(s_list)
    return s_values, all_s_values


In [96]:
time_steps = np.linspace(0, 10000, 100)
shots = 100024
num_pairs = 10
T1 = 40
T2 = 60

In [97]:
s_values, all_s_values = compute_chsh_parameter_thermal(time_steps, shots, num_pairs, T1, T2)
print(s_values)

KeyboardInterrupt: 